### Step 1: <a href='\src\agents\state.ts'>src/agents/state.ts </a>

This file acts as the "Redux Store" for our Agent. In LangGraph, the "State" is an object that every node can read and update. We define it here so all our nodes know what data they are working with.

message: To save human messages and AI Responses so every node can access 

blueprint: save the blue print currently designed by archetic node so next node can acess it

generatedCode: save the code generated by the coding node 

errors: save the errors found by Reviewer node



### Step 2: <a href='\src\agents\model.ts'>src/agents/model.ts </a>

This file initializes the Gemini model. Instead of creating a new model instance in every controller, we do it once here.

We use the ChatGoogleGenerativeAI class from LangChain.



### Step 3: <a href='\src\agents\nodes\architect.ts'>src/agents/nodes/architect.ts </a>

1. It defines a Zod Schema so the AI knows exactly what fields to fill (AWS services, VPC needs, etc.).

2. It uses a System Prompt to tell Gemini to act like a Senior Solutions Architect.
    Why use ChatPromptTemplate instead of just a raw string like 
    *Prompt=`You are a system archetic..... ${state.message}....etc`*

    Reason is that it provides different roles and a cleaner way to communicate with LLMs
    
    1. System Role: Instructions for the AI itself. “You are an AWS architect, follow these rules, output JSON only.” The AI always sees this first.

    2. User/Placeholder: Input from the user or frontend. Can be dynamic. {messages} is a placeholder for this content.

    3. AI/Assistant: (Optional) Previous AI responses, if you want to maintain conversation history.
    

    It allows us to create chains using .pipe where ouptput of one model becomes input of next model, chainig can be done without usnig the prompttemplate but things becoe messy.

3. It returns the updated blueprint to the AgentState and a message for user.



### Step 4: <a href='\src\agents\graph.ts'>src/agents/graph.ts </a>

This is where we "wire" the nodes together.

### Step 5: <a href='\src\controller\agentController.ts'>src/controller/agentController.ts </a>

Now we actually trigger the graph from our Express controller.

We get a request on a route which contains a user prompt. 

we call the graph and pass an inital messages array which contains the user prompt.

Now graph is already using the AgentState so it updates the messages, then first node is to called archtect node so it calls that and as grpahs has the state so now archetic node can acess the user message so it process that and return a blueprint and message so both the fields of agent state are update.

